##Amazon Data

In [6]:
### Installing packages/dependencies and libraries
!pip install requests Pillow pandas tqdm --quiet

import pandas as pd
import numpy as np
import json
import os
import requests
import re
from PIL import Image
from io import BytesIO
from tqdm import tqdm
from google.colab import files

In [7]:
# ── Upload Dataset
print("Please upload your Amazon Products CSV file...")
uploaded = files.upload()

# Get the filename of whatever was uploaded
filename = list(uploaded.keys())[0]
print(f"✅ Uploaded: {filename}")

Please upload your Amazon Products CSV file...


Saving amazonProduct-data.csv to amazonProduct-data.csv
✅ Uploaded: amazonProduct-data.csv


In [8]:
# ── Load and Inspect Dataset
df = pd.read_csv(filename)

print(f"Dataset shape : {df.shape}")
print(f"\nColumns available:")
for col in df.columns:
    non_null = df[col].notna().sum()
    pct      = round(non_null / len(df) * 100, 1)
    print(f"  {col:<25} {non_null:>6} non-null ({pct}%)")

print(f"\nSample row:")
print(df.iloc[0].to_dict())

Dataset shape : (10002, 28)

Columns available:
  Uniq Id                    10002 non-null (100.0%)
  Product Name               10002 non-null (100.0%)
  Brand Name                     0 non-null (0.0%)
  Asin                           0 non-null (0.0%)
  Category                    9172 non-null (91.7%)
  Upc Ean Code                  34 non-null (0.3%)
  List Price                     0 non-null (0.0%)
  Selling Price               9895 non-null (98.9%)
  Quantity                       0 non-null (0.0%)
  Model Number                8230 non-null (82.3%)
  About Product               9729 non-null (97.3%)
  Product Specification       8370 non-null (83.7%)
  Technical Details           9212 non-null (92.1%)
  Shipping Weight             8864 non-null (88.6%)
  Product Dimensions           479 non-null (4.8%)
  Image                      10002 non-null (100.0%)
  Variants                    2478 non-null (24.8%)
  Sku                            0 non-null (0.0%)
  Product Url       

Dropping the columns with 0 non-null observations

In [9]:
# Check before dropping
print("Columns with 0 non-null values:")
null_cols = df.columns[df.notna().sum() == 0].tolist()
print(null_cols)
print(f"\nTotal columns to drop: {len(null_cols)}")

# Drop them
df = df.dropna(axis=1, how="all")

print(f"Dropped {len(null_cols)} empty columns")
print(f"\nRemaining columns ({len(df.columns)}):")
for col in df.columns:
    non_null = df[col].notna().sum()
    pct      = round(non_null / len(df) * 100, 1)
    print(f"  {col:<30} {non_null:>6} non-null ({pct}%)")

Columns with 0 non-null values:
['Brand Name', 'Asin', 'List Price', 'Quantity', 'Sku', 'Stock', 'Product Details', 'Dimensions', 'Color', 'Ingredients', 'Direction To Use', 'Size Quantity Variant', 'Product Description']

Total columns to drop: 13
Dropped 13 empty columns

Remaining columns (15):
  Uniq Id                         10002 non-null (100.0%)
  Product Name                    10002 non-null (100.0%)
  Category                         9172 non-null (91.7%)
  Upc Ean Code                       34 non-null (0.3%)
  Selling Price                    9895 non-null (98.9%)
  Model Number                     8230 non-null (82.3%)
  About Product                    9729 non-null (97.3%)
  Product Specification            8370 non-null (83.7%)
  Technical Details                9212 non-null (92.1%)
  Shipping Weight                  8864 non-null (88.6%)
  Product Dimensions                479 non-null (4.8%)
  Image                           10002 non-null (100.0%)
  Variants      

In [10]:
df.head()

,Uniq Id,Product Name,Category,Upc Ean Code,Selling Price,Model Number,About Product,Product Specification,Technical Details,Shipping Weight,Product Dimensions,Image,Variants,Product Url,Is Amazon Seller
0,4c69b61db1fc16e7013b43fc926e502d,"DB Longboards CoreFlex Crossbow 41"" Bamboo Fib...",Sports & Outdoors | Outdoor Recreation | Skate...,NaN,$237.68,NaN,Make sure this fits by entering your model num...,Shipping Weight: 10.7 pounds (View shipping ra...,NaN,10.7 pounds,NaN,https://images-na.ssl-images-amazon.com/images...,https://www.amazon.com/DB-Longboards-CoreFlex-...,https://www.amazon.com/DB-Longboards-CoreFlex-...,Y
1,66d49bbed043f5be260fa9f7fbff5957,"Electronic Snap Circuits Mini Kits Classpack, ...",Toys & Games | Learning & Education | Science ...,NaN,$99.95,55324,Make sure this fits by entering your model num...,Product Dimensions: 14.7 x 11.1 x 10.2...,The snap circuits mini kits classpack provides...,4 pounds,14.7 x 11.1 x 10.2 inches 4.06 pounds,https://images-na.ssl-images-amazon.com/images...,NaN,https://www.amazon.com/Electronic-Circuits-Cla...,Y
2,2c55cae269aebf53838484b0d7dd931a,3Doodler Create Flexy 3D Printing Filament Ref...,Toys & Games | Arts & Crafts | Craft Kits,NaN,$34.99,NaN,Make sure this fits by entering your model num...,ProductDimensions:10.3x3.4x0.8inches|ItemWeigh...,show up to 2 reviews by default No longer are ...,12.8 ounces,NaN,https://images-na.ssl-images-amazon.com/images...,NaN,https://www.amazon.com/3Doodler-Plastic-Innova...,Y
3,18018b6bc416dab347b1b7db79994afa,Guillow Airplane Design Studio with Travel Cas...,Toys & Games | Hobbies | Models & Model Kits |...,NaN,$28.91,142,Make 8 different Planes at one time. | Experim...,ProductDimensions:3.5x6.2x13inches|ItemWeight:...,Go to your orders and start the return Select ...,13.4 ounces,NaN,https://images-na.ssl-images-amazon.com/images...,NaN,https://www.amazon.com/Guillow-Airplane-Design...,Y
4,e04b990e95bf73bbe6a3fa09785d7cd0,Woodstock- Collage 500 pc Puzzle,Toys & Games | Puzzles | Jigsaw Puzzles,NaN,$17.49,62151,Make sure this fits by entering your model num...,ProductDimensions:1.9x8x10inches|ItemWeight:13...,show up to 2 reviews by default 100% Officiall...,13.4 ounces,NaN,https://images-na.ssl-images-amazon.com/images...,NaN,https://www.amazon.com/Woodstock-Collage-500-p...,Y


In [11]:
# Category analysis
print("Available categories:")
print(df["Category"].value_counts())

Available categories:
Category
Toys & Games | Games & Accessories | Board Games                                                                                                          284
Toys & Games | Puzzles | Jigsaw Puzzles                                                                                                                   274
Toys & Games | Stuffed Animals & Plush Toys | Stuffed Animals & Teddy Bears                                                                               252
Toys & Games | Toy Figures & Playsets | Action Figures                                                                                                    235
Toys & Games | Dolls & Accessories | Dolls                                                                                                                193
                                                                                                                                                         ... 
Clothing, Shoes & Jew

In [12]:
print(df.columns.tolist())

['Uniq Id', 'Product Name', 'Category', 'Upc Ean Code', 'Selling Price', 'Model Number', 'About Product', 'Product Specification', 'Technical Details', 'Shipping Weight', 'Product Dimensions', 'Image', 'Variants', 'Product Url', 'Is Amazon Seller']


In [13]:
import pandas as pd

# ── Define Optimal Attribute Combination ──────────────────────────────────────
# This satisfies the task requirement of defining the right combination
# of product attributes for comprehensive product descriptions

SELECTED_ATTRIBUTES = {
    "Product Name",
    "Category",
    "Selling Price",
    "About Product",
    "Product Specification",
    "Technical Details",
    "Shipping Weight",
    "Product Dimensions",
    "Image",
    "Variants",
}

EXCLUDED_ATTRIBUTES = {
  "Product Url",
  "Asin",
  "Upc Ean Code",
  "Uniq Id",
  "Is Amazon Seller",
  "Model Number",
}

print("Selected attributes:")
for col in SELECTED_ATTRIBUTES:
    print(f"  ✅ {col:<20} — Selected")

print("\nExcluded attributes:")
for col in EXCLUDED_ATTRIBUTES:
    print(f"  ❌ {col:<20} — Excluded")

Selected attributes:
  ✅ Image                — Selected
  ✅ Shipping Weight      — Selected
  ✅ Technical Details    — Selected
  ✅ Product Name         — Selected
  ✅ About Product        — Selected
  ✅ Product Specification — Selected
  ✅ Selling Price        — Selected
  ✅ Variants             — Selected
  ✅ Product Dimensions   — Selected
  ✅ Category             — Selected

Excluded attributes:
  ❌ Is Amazon Seller     — Excluded
  ❌ Upc Ean Code         — Excluded
  ❌ Product Url          — Excluded
  ❌ Model Number         — Excluded
  ❌ Asin                 — Excluded
  ❌ Uniq Id              — Excluded


In [14]:
# ── Data Cleaning and Consistency Checks ─────────────────────
def clean_and_validate_product(row) -> dict | None:
    """
    Validate and clean a product row using all selected attributes.
    Returns None if product doesn't meet quality threshold.
    """
    product_name_col       = row.get("Product Name")
    about_col              = row.get("About Product")
    image_col              = row.get("Image")
    price_col              = row.get("Selling Price")
    category_col           = row.get("Category")
    technical_details_col  = row.get("Technical Details")
    product_spec_col       = row.get("Product Specification")
    product_dims_col       = row.get("Product Dimensions")
    shipping_weight_col    = row.get("Shipping Weight")
    variants_col           = row.get("Variants")

    # 1. Must have a name
    if pd.isna(product_name_col) or str(product_name_col).strip() == "":
        return None

    # 2. Must have at least description or image
    has_description = (
        pd.notna(about_col) and
        len(str(about_col).strip()) > 20
    )
    has_image = (
        pd.notna(image_col) and
        str(image_col).strip().startswith("http")
    )

    if not has_description and not has_image:
        return None

    # ── Clean price
    price = str(price_col if pd.notna(price_col) else "N/A")
    price = re.sub(r"[₹$,]", "", price).strip()
    price = price if price and price != "nan" else "N/A"

    # ── Clean about product
    about = str(about_col if pd.notna(about_col) else "").strip()
    about = re.sub(r"\s+", " ", about)

    # ── Clean technical details
    technical = str(technical_details_col if pd.notna(technical_details_col) else "").strip()
    technical = re.sub(r"\s+", " ", technical)
    technical = technical if technical and technical != "nan" else ""

    # ── Clean product specification
    specification = str(product_spec_col if pd.notna(product_spec_col) else "").strip()
    specification = re.sub(r"\s+", " ", specification)
    specification = specification if specification and specification != "nan" else ""

    # ── Clean product dimensions ──────────────────────────────────────────────
    dimensions = str(product_dims_col if pd.notna(product_dims_col) else "").strip()
    dimensions = dimensions if dimensions and dimensions != "nan" else ""

    # ── Clean shipping weight ─────────────────────────────────────────────────
    shipping_weight = str(shipping_weight_col if pd.notna(shipping_weight_col) else "").strip()
    shipping_weight = shipping_weight if shipping_weight and shipping_weight != "nan" else ""

    # ── Clean variants ────────────────────────────────────────────────────────
    variants = str(variants_col if pd.notna(variants_col) else "").strip()
    variants = variants if variants and variants != "nan" else ""

    # ── Parse category hierarchy ──────────────────────────────────────────────
    main_category = ""
    sub_category  = ""
    full_category = ""
    if pd.notna(category_col):
        cat_parts     = [p.strip() for p in str(category_col).split("|")]
        cat_parts     = [p for p in cat_parts if p]
        main_category = cat_parts[0] if len(cat_parts) > 0 else ""
        sub_category  = cat_parts[1] if len(cat_parts) > 1 else ""
        full_category = str(category_col).strip()

    return {
        "name"          : str(product_name_col).strip(),
        "category"      : main_category,
        "sub_category"  : sub_category,
        "full_category" : full_category,
        "price"         : price,
        "about"         : about,
        "technical"     : technical,
        "specification" : specification,
        "dimensions"    : dimensions,
        "shipping_weight": shipping_weight,
        "variants"      : variants,
        "image_url"     : str(image_col if pd.notna(image_col) else "").strip(),
        "has_image"     : has_image,
        "has_description": has_description,
    }


def build_product_description(product: dict) -> str:
    """
    Build comprehensive product description from all selected attributes.

    Attribute order rationale:
    1. Product Name      — most important for identification
    2. Category          — context for what type of product it is
    3. Price             — frequently queried
    4. About Product     — feature bullets, highly descriptive
    5. Variants          — color/size options customers ask about
    6. Technical Details — specs for technical queries
    7. Product Spec      — additional spec context
    8. Dimensions        — physical size queries
    9. Shipping Weight   — practical info customers ask about
    """
    parts = []

    # 1. Product name — always first
    parts.append(f"Product: {product['name']}")

    # 2. Category — full hierarchy for context
    if product["category"]:
        cat_str = product["category"]
        if product["sub_category"]:
            cat_str += f" > {product['sub_category']}"
        parts.append(f"Category: {cat_str}")

    # 3. Price
    if product["price"] and product["price"] != "N/A":
        parts.append(f"Price: ${product['price']}")

    # 4. About product — feature bullets (truncated to 400 chars)
    if product["about"] and len(product["about"]) > 10:
        parts.append(f"Features: {product['about'][:400]}")

    # 5. Variants — colors, sizes etc (truncated to 200 chars)
    if product["variants"] and len(product["variants"]) > 5:
        parts.append(f"Variants: {product['variants'][:200]}")

    # 6. Technical details (truncated to 300 chars)
    if product["technical"] and len(product["technical"]) > 10:
        parts.append(f"Technical Details: {product['technical'][:300]}")

    # 7. Product specification (truncated to 300 chars)
    if product["specification"] and len(product["specification"]) > 10:
        parts.append(f"Specifications: {product['specification'][:300]}")

    # 8. Product dimensions
    if product["dimensions"]:
        parts.append(f"Dimensions: {product['dimensions']}")

    # 9. Shipping weight
    if product["shipping_weight"]:
        parts.append(f"Shipping Weight: {product['shipping_weight']}")

    return "\n".join(parts)


# ── Process sampled products ───────────────────────────────────────────────────
df_sample = df.sample(n=1000, random_state=42)

products = []
skipped  = 0

for _, row in tqdm(df_sample.iterrows(), total=len(df_sample), desc="Processing"):
    cleaned = clean_and_validate_product(row)
    if cleaned:
        cleaned["text"] = build_product_description(cleaned)
        products.append(cleaned)
    else:
        skipped += 1

print(f"\n✅ Processing complete")
print(f"   Valid products : {len(products)}")
print(f"   Skipped        : {skipped}")
print(f"   Retention rate : {round(len(products)/len(df_sample)*100, 1)}%")

# ── Show sample descriptions ───────────────────────────────────────────────────
print(f"\nSample product description:")
print("─" * 60)
print(products[0]["text"])
print("─" * 60)

# Show which attributes were populated
print(f"\nAttribute coverage in processed products:")
df_products = pd.DataFrame(products)
for col in ["about", "technical", "specification", "dimensions",
            "shipping_weight", "variants", "has_image"]:
    if col in df_products.columns:
        if col in ["has_image", "has_description"]:
            count = df_products[col].sum()
        else:
            count = (df_products[col] != "").sum()
        pct = round(count / len(df_products) * 100, 1)
        print(f"  {col:<20} {count:>5} ({pct}%)")

Processing: 100%|██████████| 1000/1000 [00:00<00:00, 5724.72it/s]


✅ Processing complete
   Valid products : 1000
   Skipped        : 0
   Retention rate : 100.0%

Sample product description:
────────────────────────────────────────────────────────────
Product: Tamiya 35101 1/35 German Flkpnzr Mobelwagen
Category: Toys & Games > Toy Figures & Playsets
Price: $29.80
Features: Make sure this fits by entering your model number. | Tamiya Item#: 35101 | Precision **ASSEMBLY-REQUIRED PLASTIC MODEL KIT** with parts mounted on sprue trees | For ages 10 and older; To avoid choking or injury, keep all model kit parts away from small children | Assembly & painting required. Requires glue, paint, & modeling tools (not included) | For damaged product & defective part, contact Tami
Technical Details: show up to 2 reviews by default Highly detailed 1/35 scale model kit of the World War II German 3. 7cm FlaK auf Fahrgestell Panzerkampfwagen IV, nicknamed Mobelwagen or "Furniture Van". The Mobelwagen was essentially a Panzer IV chassis fitted with an open-top superst

In [15]:
# ── Data Quality Summary
df_products = pd.DataFrame(products)

print("Data Quality Summary")
print("=" * 50)
print(f"Total products       : {len(df_products)}")
print(f"With images          : {df_products['image_url'].apply(lambda x: x.startswith('http')).sum()}")
print(f"With descriptions    : {df_products['about'].apply(lambda x: len(x) > 20).sum()}")
print(f"With prices          : {(df_products['price'] != 'N/A').sum()}")
print(f"\nCategory distribution:")
print(df_products["category"].value_counts())
print(f"\nAll unique categories:\n{df_products['category'].unique().tolist()}")
print(f"\nAvg description length: {df_products['about'].apply(len).mean():.0f} chars")

# Save processed products
with open("amazon_products_processed.json", "w", encoding="utf-8") as f:
    json.dump(products, f, ensure_ascii=False, indent=2)

print(f"\nSaved: amazon_products_processed.json")
files.download("amazon_products_processed.json")

Data Quality Summary
Total products       : 1000
With images          : 1000
With descriptions    : 977
With prices          : 989

Category distribution:
category
Toys & Games                 681
                              86
Sports & Outdoors             63
Home & Kitchen                63
Clothing, Shoes & Jewelry     51
Baby Products                 20
Arts, Crafts & Sewing         10
Industrial & Scientific        7
Office Products                6
Beauty & Personal Care         3
Grocery & Gourmet Food         2
Hobbies                        2
Cell Phones & Accessories      1
Patio, Lawn & Garden           1
Automotive                     1
Pet Supplies                   1
Tools & Home Improvement       1
Health & Household             1
Name: count, dtype: int64

All unique categories:
['Toys & Games', '', 'Sports & Outdoors', 'Home & Kitchen', 'Clothing, Shoes & Jewelry', 'Industrial & Scientific', 'Office Products', 'Beauty & Personal Care', 'Grocery & Gourmet Food', 'Baby

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## CLIP

In [1]:
# SECTION 1: INSTALL DEPENDENCIES
!pip install transformers torch torchvision --quiet
!pip install --upgrade pyarrow --quiet
!pip install --quiet --upgrade langchain langchain-community langchain-chroma \
    langchain-openai sentence-transformers chromadb datasets \
    presidio-analyzer presidio-anonymizer spacy huggingface_hub openai
!pip install -U langchain-text-splitters langchain-huggingface
!pip install --upgrade opentelemetry-api opentelemetry-sdk --quiet
!python -m spacy download en_core_web_lg --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 19.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.9/43.9 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.7/163.7 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 75.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.8/125.8 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 51.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 58.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 41.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 266.3/266.3 kB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 842.9/842.9 kB 50.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 84.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
# ── SECTION 2: IMPORTS
from transformers import CLIPModel, CLIPProcessor
from PIL import Image
import requests
from io import BytesIO
import numpy as np
import torch
import chromadb
import os
import json
import gc
import pandas as pd
from tqdm import tqdm
from google.colab import userdata

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from sentence_transformers import CrossEncoder
from presidio_analyzer import AnalyzerEngine
from presidio_anonymizer import AnonymizerEngine
from huggingface_hub import login
from openai import OpenAI

print("All imports successful.")

/tmp/ipykernel_2614/3665924231.py:19: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.chat_message_histories import ChatMessageHistory


All imports successful.


In [3]:
# ── SECTION 3: API KEYS
# Required: HF_TOKEN from huggingface.co/settings/tokens (free account)
# Optional: OPENAI_API_KEYS only needed for RAGAS evaluation section

login(token=userdata.get("HF_TOKEN"))
# os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEYS")  # only for RAGAS eval
print("API keys set.")

API keys set.


In [4]:
import requests
from io import BytesIO

# ── SECTION 4: CLIP MODEL
print("Loading CLIP model...")
device         = "cuda" if torch.cuda.is_available() else "cpu"
clip_model     = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model.eval()
print(f"CLIP loaded on {device}")


def get_text_embedding(text: str) -> list[float]:
    inputs = clip_processor(
        text=[text],
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=77,
    ).to(device)
    with torch.no_grad():
        raw = clip_model.get_text_features(**inputs)
        # Handle both transformer versions:
        # older versions return a plain Tensor
        # some versions return a ModelOutput object
        if isinstance(raw, torch.Tensor):
            emb = raw
        else:
            emb = raw.pooler_output  # extract tensor from ModelOutput
        emb = emb / emb.norm(dim=-1, keepdim=True)
    return emb.cpu().numpy()[0].tolist()


def get_image_embedding(image: Image.Image) -> list[float]:
    inputs = clip_processor(
        images=image,
        return_tensors="pt",
    ).to(device)
    with torch.no_grad():
        raw = clip_model.get_image_features(**inputs)
        if isinstance(raw, torch.Tensor):
            emb = raw
        else:
            emb = raw.pooler_output
        emb = emb / emb.norm(dim=-1, keepdim=True)
    return emb.cpu().numpy()[0].tolist()

def load_image(url: str) -> Image.Image | None:
    try:
        resp = requests.get(url, timeout=10, headers={"User-Agent": "Mozilla/5.0"})
        resp.raise_for_status()
        return Image.open(BytesIO(resp.content)).convert("RGB")
    except Exception:
        return None

def get_combined_embedding(text: str, image_url: str) -> list[float]:
    """Average text and image embeddings for multimodal representation."""
    text_emb = get_text_embedding(text)
    image    = load_image(image_url)
    if image:
        image_emb = get_image_embedding(image)
        return np.mean([text_emb, image_emb], axis=0).tolist()
    return text_emb  # fallback to text only


print("Embedding functions defined.")

Loading CLIP model...


config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  605MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  605MB            

...

vocab.json:   0%|          | 0.00/862k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.22M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

CLIP loaded on cuda
Embedding functions defined.


In [16]:
import pandas as pd
import json
from tqdm import tqdm
import chromadb

# ── SECTION 5: CHROMADB — BUILD INDEX

# 1. Load Product Data:
# This block loads the product data from 'amazon_products_processed.json'.
# Removed the 'if "products" not in dir():' check to ensure data is always reloaded.
with open("amazon_products_processed.json", "r") as f:
    products = json.load(f)

# 2. Initialize ChromaDB Client:
# A persistent ChromaDB client is created, meaning the database will be stored
# on disk at the specified path ('amazon_chroma_db') and will persist across sessions.
chroma_client = chromadb.PersistentClient(path="amazon_chroma_db")

# 3. Handle Existing Collection:
# This `try-except` block attempts to delete an existing collection named 'amazon_products'.
# This is useful for development, ensuring a clean slate each time the index is built.
# If the collection doesn't exist, it simply skips the deletion without error.
try:
    chroma_client.delete_collection("amazon_products")
    print("Existing collection cleared.")
except Exception:
    pass

# 4. Create New Collection:
# A new collection named 'amazon_products' is created. The 'hnsw:space': 'cosine'
# metadata configures the similarity metric to be used for searches within this collection
# to cosine similarity, which is common for embeddings.
collection = chroma_client.create_collection(
    name="amazon_products",
    metadata={"hnsw:space": "cosine"},
)

# 5. Generate Embeddings and Add to ChromaDB:
# This is the core loop where each product is processed.
print(f"Generating embeddings for {len(products)} products...")
failed = 0

# It iterates through each 'product' in the loaded 'products' list.
# tqdm provides a progress bar for better user experience.
for i, product in tqdm(enumerate(products), total=len(products)):
    try:
        # 'get_combined_embedding' (defined in Section 4) generates a multimodal embedding
        # by combining text and image embeddings for the product.
        embedding = get_combined_embedding(product["text"], product["image_url"])

        # The product's ID, generated embedding, text content (document),
        # and various metadata fields (name, category, price, rating, image_url)
        # are added to the ChromaDB collection.
        collection.add(
            ids=[str(i)],
            embeddings=[embedding],
            documents=[product["text"]],
            metadatas=[{
                "name"     : product["name"],
                "category" : product["category"],
                "price"    : str(product["price"]),
                "image_url": product["image_url"],
            }],
        )
    except Exception as e:
        # If any error occurs during embedding generation or adding a product (e.g., missing data),
        # it's caught here, the product is skipped, and the error is printed.
        print(f"  Skipping product {i}: {e}")
        failed += 1

# After the loop, it prints a summary of how many products were successfully stored
# and how many failed.
print(f"\nStored {collection.count()} products in ChromaDB.")
print(f"Failed : {failed}")

# 6. Save Embeddings to CSV (for inspection/backup):
# This section retrieves all data (embeddings, documents, and metadatas) from the
# newly created ChromaDB collection.
all_data = collection.get(include=["embeddings", "documents", "metadatas"])

# It then formats this retrieved data into a list of dictionaries.
embedding_records = []
for i in range(len(all_data["ids"])):
    record = {
        "id"              : all_data["ids"][i],
        "document_content": all_data["documents"][i],
        "embedding"       : str(all_data["embeddings"][i]), # Convert embedding list to string
        **all_data["metadatas"][i], # Unpack other metadata fields
    }
    embedding_records.append(record)

# Finally, it converts this list into a Pandas DataFrame and saves it as a CSV file.
# This allows for easy inspection and external use of the indexed data.
df_embeddings = pd.DataFrame(embedding_records)
df_embeddings.to_csv("amazon_embeddings_with_metadata.csv", index=False)
print(f"Embeddings saved. Shape: {df_embeddings.shape}")

Generating embeddings for 1000 products...


100%|██████████| 1000/1000 [06:06<00:00,  2.73it/s]



Stored 1000 products in ChromaDB.
Failed : 0
Embeddings saved. Shape: (1000, 7)


In [17]:
import shutil
shutil.make_archive("amazon_chroma_db", "zip", "amazon_chroma_db")

from google.colab import files
files.download("amazon_chroma_db.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ── SECTION 6: RERANKER
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
print("Reranker loaded.")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Reranker loaded.


In [ ]:
# ── SECTION 7: RETRIEVAL ──────────────────────────────────────────────────────

def retrieve_products(
    query_text  : str         = None,
    query_image : Image.Image = None,
    top_k       : int         = 5,
) -> list[dict]:
    """Retrieve relevant products using text, image, or both."""

    n_candidates = min(20, collection.count())
    if n_candidates == 0:
        return []

    if query_text and query_image:
        text_emb  = get_text_embedding(query_text)
        image_emb = get_image_embedding(query_image)
        query_emb = np.mean([text_emb, image_emb], axis=0).tolist()
    elif query_image:
        query_emb = get_image_embedding(query_image)
    else:
        query_emb = get_text_embedding(query_text)

    results = collection.query(
        query_embeddings=[query_emb],
        n_results=n_candidates,
        include=["documents", "metadatas", "distances"],
    )

    candidates = list(zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0],
    ))

    if query_text and candidates:
        pairs  = [[query_text, doc] for doc, _, _ in candidates]
        scores = reranker.predict(pairs)
        ranked = sorted(zip(scores, candidates), key=lambda x: x[0], reverse=True)
        candidates = [c for _, c in ranked]

    return [
        {
            "text"      : doc,
            "metadata"  : meta,
            "similarity": round(1 - dist, 4),
        }
        for doc, meta, dist in candidates[:top_k]
    ]

# Quick retrieval test
test_queries = [
    "Samsung Galaxy smartphone features",
    "wireless earbuds noise cancellation",
    "laptop with good battery life",
]

for query in test_queries:
    print(f"\nQuery: '{query}'")
    print("─" * 60)
    results = retrieve_products(query_text=query, top_k=3)
    for i, r in enumerate(results, 1):
        print(f"[{i}] (sim: {r['similarity']}) {r['metadata']['name'][:60]}")
        print(f"     Price: {r['metadata']['price']}")


Query: 'Samsung Galaxy smartphone features'
────────────────────────────────────────────────────────────
[1] (sim: 0.4875) DJI Smart Controller
     Price: 665.57
[2] (sim: 0.4715) E-flite EC5 Device Connector (2), EFLAEC501
     Price: 9.31
[3] (sim: 0.4639) Little Pretender Walkie Talkies for Kids, 2 Mile Range, 3 Ch
     Price: 26.99

Query: 'wireless earbuds noise cancellation'
────────────────────────────────────────────────────────────
[1] (sim: 0.5216) Turtle Beach XO Three Gaming Headset for Xbox One
     Price: 53 98 59.95 #listPriceLegalMessageText { margin-left: 4px !important; } #listPriceLegalMessage .a-popover-trigger:hover { text-decoration: none !important; } #listPriceLegalMessage .a-icon-popover { display: none !important; margin-left: 0px !important; margin-top: 6px !important; } Save 5.97 (10%)
[2] (sim: 0.5145) E-flite EC5 Device Connector (2), EFLAEC501
     Price: 9.31
[3] (sim: 0.4503) E-flite 7.6-Gram DS76 Digital Sub-Micro Servo
     Price: 21.99

Query: 'lap

In [ ]:
eval_questions = [
    'What is a good board game for family game night?',
    'Recommend a toy for a 5 year old child',
    'What are the best action figures available?',
    'Can you suggest a puzzle for adults?',
    'What stuffed animals are good for toddlers?',
    'What outdoor toys are available for kids?',
    'Recommend an arts and crafts kit for children',
    'What are some educational toys for kids?',
    'What card games are good for parties?',
    'Recommend a building set for creative play',
]

recall_test_queries = [
    {"query": "gaming headset for xbox",          "relevant_product": "Turtle Beach"},
    {"query": "drone controller",                 "relevant_product": "DJI"},
    {"query": "educational toy for kids",         "relevant_product": "Learning"},
    {"query": "wrist protection for skating",     "relevant_product": "Wrist"},
    {"query": "classic memory game",              "relevant_product": "Simon"},
]

# Pre-compute eval retrievals
retrieved_results = {q: retrieve_products(query_text=q, top_k=5) for q in eval_questions}
print(f"Retrieval pre-computed for {len(eval_questions)} questions ✓")

# Pre-compute recall retrievals — MUST happen here before CLIP is freed
recall_retrieved = {
    item["query"]: retrieve_products(query_text=item["query"], top_k=10)
    for item in recall_test_queries
}
print(f"Recall retrieval pre-computed for {len(recall_test_queries)} queries ✓")

print("\n=== RECALL DEBUG: What was actually retrieved? ===")
for item in recall_test_queries:
    print(f"\nQuery: '{item['query']}' | Looking for: '{item['relevant_product']}'")
    retrieved = recall_retrieved[item["query"]]
    if not retrieved:
        print("  ⚠ NO RESULTS RETURNED — collection may be empty or CLIP failed")
    for r in retrieved[:3]:
        name = r["metadata"]["name"]
        match = item["relevant_product"].lower() in name.lower()
        print(f"  {'✅' if match else '❌'} {name[:70]}")

# NOW free CLIP — everything needed has been retrieved
del clip_model
gc.collect()
torch.cuda.empty_cache()
print(f"CLIP freed. GPU: {torch.cuda.memory_allocated()/1e9:.2f} GB")

Retrieval pre-computed for 10 questions ✓
Recall retrieval pre-computed for 5 queries ✓

=== RECALL DEBUG: What was actually retrieved? ===

Query: 'gaming headset for xbox' | Looking for: 'Turtle Beach'
  ✅ Turtle Beach XO Three Gaming Headset for Xbox One
  ❌ Winning Moves Games Triple Cross
  ❌ dreamGEAR My Arcade Learning Pad Educational Toy: 70 Brain and Puzzle 

Query: 'drone controller' | Looking for: 'DJI'
  ✅ MightySkins Skin Compatible with DJI Mavic Air Drone - Groovy 60s | Mi
  ❌ Airdog Front Left Quadcopter Arm
  ✅ MightySkins Skin Compatible with DJI Spark Mini Drone Combo - Style | 

Query: 'educational toy for kids' | Looking for: 'Learning'
  ✅ dreamGEAR My Arcade Learning Pad Educational Toy: 70 Brain and Puzzle 
  ❌ hand2mind Pop-Up Center, Math Games With Ten-frame (Ages 5+) - 10 Crit
  ❌ Didax Educational Resources Developing Cutting Skills: Early Years

Query: 'wrist protection for skating' | Looking for: 'Wrist'
  ❌ Atom Elite Palm Guard
  ✅ Rector Proformer Wris

In [ ]:
# ── SECTION 8: LLM — MISTRAL VIA HUGGINGFACE (FREE) ─────────────────────────
#
# Uses HuggingFace Inference API — no credit card required.
# Sign up at huggingface.co, generate a token, store as HF_TOKEN in Colab secrets.
# Model: mistralai/Mistral-7B-Instruct-v0.3
#
# NOTE: If you hit rate limits on the free tier, swap the repo_id below to:
#   "HuggingFaceH4/zephyr-7b-beta"   — another good free option
#   "meta-llama/Meta-Llama-3-8B-Instruct" — requires accepting license on HF

llm_endpoint = HuggingFaceEndpoint(
    repo_id                  = "meta-llama/Meta-Llama-3-8B-Instruct",
    huggingfacehub_api_token = userdata.get("HF_TOKEN"),
    task                     = "text-generation",
    max_new_tokens           = 300,
    temperature              = 0.2,
    do_sample                = True,
)

# Wrap in ChatHuggingFace so it's compatible with LangChain chat prompt templates
llm = ChatHuggingFace(llm=llm_endpoint)
print("LLM ready: Meta-Llama-3-8B-Instruct via HuggingFace Inference API")

LLM ready: Meta-Llama-3-8B-Instruct via HuggingFace Inference API


In [ ]:
# ── SECTION 9: PROMPT TEMPLATE ────────────────────────────────────────────────

prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a helpful product assistant for an e-commerce platform.
Answer customer questions about products using ONLY the provided context.

--- Few-Shot Examples ---
   Zero-shot example:
Q: What are the features of this product?
A: Based on the product information, [specific features from context].

One-shot example:
Q: What are the features of Monopoly?
A: Monopoly is a classic board game for 2-6 players featuring property trading,
   buying houses and hotels, and collecting rent.

Two-shot example:
Q: Compare Lego City and Lego Technic.
A: Lego City targets younger children (5+) with everyday scenarios, while Lego
   Technic targets older builders (10+) with complex mechanical models.

Q: Can you identify this product from the image?
A: This appears to be a building set. It is used for creative construction play
   and comes with multiple pieces for assembling various structures.
--- End Examples ---

CRITICAL Guidelines:
- Answer DIRECTLY and SPECIFICALLY — lead with the answer in your first sentence.
- Use ONLY the provided product context — never fabricate specs, prices, or features.
- Keep answers focused and concise — 2 to 4 sentences.
- For image questions, describe the product and its key features.
- If the context does not contain the answer, say:
  "I don't have that information. Please visit the official product page or be more specific."
- Always provide the source URL when available.
- Be concise, professional, and friendly.

Context:
{context}"""),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}"),
])

In [ ]:
# SECTION 10: SESSION MEMORY

store = {}

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

SESSION_ID = "amazon_session"
print("Session memory ready.")

Session memory ready.


In [ ]:
# ── SECTION 11: RESPONSIBLE AI UTILITIES ─────────────────────────────────────

analyzer   = AnalyzerEngine()
anonymizer = AnonymizerEngine()


def redact_pii(text: str) -> str:
    results = analyzer.analyze(text=text, language="en")
    if not results:
        return text
    return anonymizer.anonymize(text=text, analyzer_results=results).text


def check_hallucination(answer: str, contexts: list[str]) -> dict:
    """
    Lightweight grounding check: measures what fraction of 3-word phrases
    in the answer also appear in the retrieved context.
    Score >= 0.3 is considered grounded.
    """
    context = " ".join(contexts).lower()
    words   = answer.lower().split()
    phrases = [" ".join(words[i:i+3]) for i in range(len(words) - 2)]
    if not phrases:
        return {"grounded": True, "score": 1.0, "warning": None}
    score   = round(sum(1 for p in phrases if p in context) / len(phrases), 2)
    warning = "⚠ Answer may contain info not found in source documents." if score < 0.3 else None
    return {"grounded": score >= 0.3, "score": score, "warning": warning}


BLOCKED_TOPICS = ["politics", "religion", "violence", "illegal", "drugs", "hate"]

def is_appropriate(query: str) -> bool:
    return not any(t in query.lower() for t in BLOCKED_TOPICS)

In [ ]:
# ── SECTION 12: MAIN RAG QUERY FUNCTION
def run_rag_query(
    question         : str         = None,
    query_image      : Image.Image = None,
    session_id       : str         = SESSION_ID,
    precomputed_products : list    = None,  # pass pre-retrieved results to skip CLIP
) -> dict:
    """
    Full multimodal RAG pipeline.
    precomputed_products: if provided, skips retrieval (used when CLIP is freed).
    """
    clean_question = redact_pii(question).strip() if question else ""

    if clean_question and not is_appropriate(clean_question):
        return {
            "question"    : question,
            "answer"      : "I can't respond to that type of question.",
            "contexts"    : [],
            "products"    : [],
            "sources"     : [],
            "grounded"    : None,
            "ground_score": None,
            "warning"     : None,
        }

    # Use pre-computed results if available (post CLIP-free), else retrieve live
    products = precomputed_products if precomputed_products is not None else retrieve_products(
        query_text  = clean_question or None,
        query_image = query_image,
        top_k       = 5,
    )

    context = "\n\n".join([
        f"[Product {i+1}]\n{p['text']}"
        for i, p in enumerate(products)
    ])

    if query_image and clean_question:
        input_text = f"{clean_question} [User also uploaded a product image]"
    elif query_image:
        input_text = "Can you identify this product and describe its features and usage?"
    else:
        input_text = clean_question

    chain = (
        RunnablePassthrough.assign(context=lambda _: context)
        | prompt
        | llm
        | StrOutputParser()
    )

    qa_chain_run = RunnableWithMessageHistory(
        chain,
        get_session_history,
        input_messages_key="input",
        history_messages_key="chat_history",
    )

    answer = qa_chain_run.invoke(
        {"input": input_text},
        config={"configurable": {"session_id": session_id}},
    )

    contexts      = [p["text"] for p in products]
    hallucination = check_hallucination(answer, contexts)
    sources       = [
        p["metadata"].get("image_url", "")
        for p in products
        if p["metadata"].get("image_url")
    ]

    return {
        "question"    : question,
        "answer"      : answer,
        "contexts"    : contexts,
        "products"    : products,
        "sources"     : sources,
        "grounded"    : hallucination["grounded"],
        "ground_score": hallucination["score"],
        "warning"     : hallucination["warning"],
    }

# Quick smoke test
# Update question_for_smoke_test to match the new eval_questions in cell 0XuyHrnOSoLz
question_for_smoke_test = eval_questions[0]
result = run_rag_query(
    question=question_for_smoke_test,
    precomputed_products=retrieved_results[question_for_smoke_test]
)
print(f"Question : {result['question']}")
print(f"Answer   : {result['answer']}")
print(f"Grounding: {result['ground_score']}")

/tmp/ipykernel_3331/103484258.py:86: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  result = run_rag_query(


Question : What is a good board game for family game night?
Answer   : Based on the provided context, I would recommend Monopoly: Fortnite Edition Board Game Inspired by Fortnite Video Game Ages 13 & Up. It's a thrilling game for 2-7 players, perfect for family game nights, where players claim locations, battle opponents, and avoid the Storm.
Grounding: 0.4


In [ ]:
answers  = []
contexts = []

for q in tqdm(eval_questions, desc="Generating"):
    # Pass pre-computed products so CLIP is not needed
    result = run_rag_query(
        question             = q,
        precomputed_products = retrieved_results[q],
        session_id           = f"eval_gen_{q[:20]}",  # unique session per query
    )
    answers.append(result["answer"])
    contexts.append(result["contexts"])

print(f"Generated {len(answers)} answers ✓")

Generating:   0%|          | 0/10 [00:00<?, ?it/s]/tmp/ipykernel_3331/3248949662.py:6: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  result = run_rag_query(
Generating: 100%|██████████| 10/10 [00:14<00:00,  1.45s/it]

Generated 10 answers ✓


In [ ]:
# After the generation loop
print(f"Generated {len(answers)} answers ✓")

# ── ADD THIS: Display answers for each eval question
print("\n" + "="*80)
print("EVAL QUESTION ANSWERS")
print("="*80)

for i, (q, a, ctx) in enumerate(zip(eval_questions, answers, contexts), 1):
    print(f"\nQ{i}: {q}")
    print(f"{'─'*60}")
    print(f"Answer: {a}")
    print(f"\nRetrieved context snippets:")
    for j, c in enumerate(ctx[:2], 1):  # show top 2 context chunks
        print(f"  [{j}] {c[:150]}...")
    print()

import warnings
warnings.filterwarnings('ignore')

!pip install ragas --quiet
# ── SECTION 13: RAGAS EVALUATION (requires OPENAI_API_KEY) ───────────
#
# RAGAS uses an LLM judge internally. We use gpt-3.5-turbo as it's cheapest.
# If you don't have an OpenAI key, skip to Section 14 (Recall@K eval).

from datasets import Dataset
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
from ragas import evaluate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEYS")
openai_client = OpenAI()

def score_metric(system_prompt: str, user_prompt: str) -> float:
    response = openai_client.chat.completions.create(
        model    = "gpt-3.5-turbo",
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
        max_tokens  = 5,
        temperature = 0,
    )
    try:
        return float(response.choices[0].message.content.strip())
    except Exception:
        return 0.0

def evaluate_rag(questions, answers, contexts):
    rows = []
    for q, a, ctx in zip(questions, answers, contexts):
        context_str = "\n".join(ctx)

        faith = score_metric(
            "You are an evaluator. Score 0.0 to 1.0 only. Output a single number.",
            f"Context:\n{context_str}\n\nAnswer:\n{a}\n\nIs the answer fully grounded in the context? Score:",
        )
        ans_rel = score_metric(
            "You are an evaluator. Score 0.0 to 1.0 only. Output a single number.",
            f"Question:\n{q}\n\nAnswer:\n{a}\n\nIs the answer relevant to the question? Score:",
        )
        ctx_rel = score_metric(
            "You are an evaluator. Score 0.0 to 1.0 only. Output a single number.",
            f"Question:\n{q}\n\nContext:\n{context_str}\n\nIs the context relevant to the question? Score:",
        )

        rows.append({
            "question"         : q,
            "faithfulness"     : faith,
            "answer_relevancy" : ans_rel,
            "context_relevance": ctx_rel,
        })
        print(f"√ {q[:60]}")

    return pd.DataFrame(rows)


results_df = evaluate_rag(eval_questions, answers, contexts)

print("\n=== Aggregate Scores ===")
for metric in ["faithfulness", "answer_relevancy", "context_relevance"]:
    print(f"{metric:<25}: {results_df[metric].mean():.4f}")

Generated 10 answers ✓

EVAL QUESTION ANSWERS

Q1: What is a good board game for family game night?
────────────────────────────────────────────────────────────
Answer: Based on the provided products, I would recommend Monopoly: Fortnite Edition Board Game Inspired by Fortnite Video Game Ages 13 & Up. It's a thrilling game for 2-7 players that's inspired by the popular Fortnite video game, making it a great option for families with older kids and teenagers.

Retrieved context snippets:
  [1] Product: Cardinal Games Jumanji The Game Action Game
Category: Toys & Games > Games & Accessories
Price: $18.35
Features: Make sure this fits by enter...
  [2] Product: Monopoly: Fortnite Edition Board Game Inspired by Fortnite Video Game Ages 13 & Up
Category: Toys & Games > Games & Accessories
Price: $15.88...


Q2: Recommend a toy for a 5 year old child
────────────────────────────────────────────────────────────
Answer: I don't have that information. Please visit the official product page or be

In [ ]:
# ── SECTION 14: RECALL@K EVALUATION

def evaluate_recall_at_k(
    test_queries : list[dict],
    k_values     : list[int] = [1, 5, 10],
) -> dict:
    """
    Calculate Recall@K.
    Each test query dict needs 'query' and 'relevant_product' keys.
    relevant_product is a substring match against retrieved product names.
    """
    scores = {f"Recall@{k}": 0 for k in k_values}
    total  = len(test_queries)

    for item in test_queries:
        retrieved = retrieve_products(
            query_text=item["query"],
            top_k=max(k_values),
        )
        for k in k_values:
            top_k_names = [r["metadata"]["name"].lower() for r in retrieved[:k]]
            if any(item["relevant_product"].lower() in name for name in top_k_names):
                scores[f"Recall@{k}"] += 1

    return {k: round(v / total, 3) for k, v in scores.items()}

# Updated recall_test_queries to match the ones used in cell 0XuyHrnOSoLz
recall_test_queries = [
    {"query": "gaming headset for xbox",          "relevant_product": "Turtle Beach"},
    {"query": "drone controller",                 "relevant_product": "DJI"},
    {"query": "educational toy for kids",         "relevant_product": "Learning"},
    {"query": "wrist protection for skating",     "relevant_product": "Wrist"},
    {"query": "classic memory game",              "relevant_product": "Simon"},
]

recall_scores = {}
for k in [1, 5, 10]:
    hits = 0
    for item in recall_test_queries:
        top_k_names = [r["metadata"]["name"].lower() for r in recall_retrieved[item["query"]][:k]]
        if any(item["relevant_product"].lower() in name for name in top_k_names):
            hits += 1
    recall_scores[f"Recall@{k}"] = round(hits / len(recall_test_queries), 3)

print("\n" + "="*50)
print("RECALL@K EVALUATION RESULTS")
print("="*50)
for metric, score in recall_scores.items():
    print(f"  {metric}: {score}")


RECALL@K EVALUATION RESULTS
  Recall@1: 0.8
  Recall@5: 1.0
  Recall@10: 1.0


In [ ]:
print("=== Per-question Breakdown ===")
cols = ["question", "faithfulness", "answer_relevancy", "context_relevance"]
print(results_df[cols].to_string(index=False))

print("\n=== Aggregate Scores ===")
for metric in ["faithfulness", "answer_relevancy", "context_relevance"]:
    print(f"{metric:<25}: {results_df[metric].mean():.4f}")

results_df.to_csv("ragas_results.csv", index=False)
print("\nSaved → ragas_results.csv")

print("\n=== Full Evaluation Summary ===")
print("Retriever : CLIP (clip-vit-base-patch32) + ChromaDB + CrossEncoder reranker")
print("Generator : Mistral-7B-Instruct-v0.3 via HuggingFace Inference API (free)")
print("Evaluator : gpt-3.5-turbo as judge (no ground truth required)")
print("Dataset   : Amazon Product Dataset 2020")
print()
print("Retrieval Metrics:")
for metric, score in recall_scores.items():
    print(f"  {metric:<12}: {score*100:.1f}%")
print()
print("Generation Metrics:")
for metric in ["faithfulness", "answer_relevancy", "context_relevance"]:
    print(f"  {metric:<25}: {results_df[metric].mean():.4f}")

#from google.colab import files
#files.download("ragas_results.csv")

=== Per-question Breakdown ===
                                        question  faithfulness  answer_relevancy  context_relevance
What is a good board game for family game night?          0.90               0.8                0.8
          Recommend a toy for a 5 year old child          0.20               0.2                0.8
     What are the best action figures available?          0.90               0.9                0.2
            Can you suggest a puzzle for adults?          0.90               0.2                0.2
     What stuffed animals are good for toddlers?          0.95               0.9                0.8
       What outdoor toys are available for kids?          1.00               1.0                0.8
   Recommend an arts and crafts kit for children          1.00               1.0                0.9
        What are some educational toys for kids?          1.00               1.0                1.0
           What card games are good for parties?          1.00       

In [ ]:
import torch
from transformers import CLIPModel, CLIPProcessor

# ── SECTION 15: FULL TEST SET EVALUATION

# Re-initialize CLIP model and processor as they were deleted in a previous cell
# This ensures that `retrieve_products` can be called if needed, or to pre-compute.
device         = "cuda" if torch.cuda.is_available() else "cpu"
clip_model     = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model.eval()
print(f"CLIP re-initialized on {device} for test set evaluation.")

amazon_test_set = [
    {"id": 1, "category": "factual",      "question": "What are the features of the DJI Smart Controller?",      "ground_truth": "The DJI Smart Controller is a remote controller compatible with DJI drones."},
    {"id": 2, "category": "factual",      "question": "What gaming headsets are available for Xbox?",            "ground_truth": "The Turtle Beach XO Three is a gaming headset designed for Xbox One."},
    {"id": 3, "category": "semantic",     "question": "Show me educational toys for kids",                       "ground_truth": "Educational toys available include learning pads with brain and puzzle activities."},
    {"id": 4, "category": "semantic",     "question": "I need something to protect my wrists while skating",     "ground_truth": "Wrist guards such as the Rector Proformer provide wrist protection."},
    {"id": 5, "category": "multi-chunk",  "question": "What RC accessories are available?",                      "ground_truth": "RC accessories include servo connectors, receivers, and digital servos from brands like E-flite and Graupner."},
    {"id": 6, "category": "multi-chunk",  "question": "What classic games are available?",                       "ground_truth": "Classic games available include Simon, a memory and pattern game."},
    {"id": 7, "category": "out-of-scope", "question": "What is the weather like today?",                         "ground_truth": "N/A — out of scope."},
    {"id": 8, "category": "out-of-scope", "question": "Who won the latest football game?",                       "ground_truth": "N/A — out of scope."},
]

with open("amazon_test_set.json", "w") as f:
    json.dump(amazon_test_set, f, indent=2)
print(f"Test set saved: {len(amazon_test_set)} questions")

# Pre-compute retrievals for the amazon_test_set using the re-initialized CLIP model
amazon_test_set_retrieved = {
    item["question"]: retrieve_products(query_text=item["question"], top_k=5)
    for item in amazon_test_set
}
print(f"Amazon test set retrieval generated for {len(amazon_test_set)} questions.")

# FIX: unique session per query — no memory bleed
test_results = []
for item in amazon_test_set:
    print(f"Running Q{item['id']}: {item['question'][:60]}...")
    # Pass pre-computed products to run_rag_query
    result = run_rag_query(
        question=item["question"],
        session_id=f"test_session_{item['id']}",
        precomputed_products=amazon_test_set_retrieved[item["question"]]
    )
    test_results.append({
        "id"          : item["id"],
        "category"    : item["category"],
        "question"    : item["question"],
        "ground_truth": item["ground_truth"],
        "answer"      : result["answer"],
        "contexts"    : result["contexts"],
        "sources"     : result.get("sources", []),
        "grounded"    : result.get("grounded"),
        "ground_score": result.get("ground_score"),
        "warning"     : result.get("warning"),
    })

print(f"\nCompleted {len(test_results)} queries.")

# Display by category
for category in ["factual", "semantic", "multi-chunk", "out-of-scope"]:
    print(f"\n{'='*70}")
    print(f"CATEGORY: {category.upper()}")
    print(f"{'='*70}")
    for r in [r for r in test_results if r["category"] == category]:
        print(f"\nQ{r['id']}: {r['question']}")
        print(f"Expected : {r['ground_truth'][:150]}")
        print(f"Got      : {r['answer'][:150]}")
        print(f"Grounding: {r['ground_score']}")
        if r["warning"]:
            print(r["warning"])

# RAGAS on in-scope questions — reuses ragas_metrics from Section 13
in_scope = [r for r in test_results if r["category"] != "out-of-scope"]

# Define ragas_metrics and metric_cols
ragas_metrics = [faithfulness, answer_relevancy, context_precision, context_recall]
metric_cols = ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]

# Instantiate Ragas-compatible LLM and Embeddings
# OPENAI_API_KEY is already set in os.environ from a previous cell
ragas_llm = LangchainLLMWrapper(ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0))
ragas_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-ada-002"))

ragas_data = {
    "question"    : [r["question"]     for r in in_scope],
    "answer"      : [r["answer"]       for r in in_scope],
    "contexts"    : [r["contexts"]     for r in in_scope],
    "ground_truth": [r["ground_truth"] for r in in_scope],
}
dataset_full = Dataset.from_dict(ragas_data)

eval_full    = evaluate(
    dataset=dataset_full,
    metrics=ragas_metrics,
    llm=ragas_llm,
    embeddings=ragas_embeddings,
    raise_exceptions=False
)

df_eval = eval_full.to_pandas()
df_eval.insert(0, "Question", [r["question"][:50] + "..." for r in in_scope])
df_eval.insert(1, "Category", [r["category"]              for r in in_scope])

for col in metric_cols:
    df_eval[col] = df_eval[col].round(3)

print("\n" + "="*80)
print("PER-QUESTION RAGAS SCORES")
print("="*80)
print(df_eval[["Question", "Category"] + metric_cols].to_string(index=False))

print("\n" + "="*60)
print("AVERAGE SCORES BY CATEGORY")
print("="*60)
print(df_eval.groupby("Category")[metric_cols].mean().round(3).to_string())

print("\n" + "="*60)
print("OVERALL AVERAGE SCORES")
print("="*60)
print(df_eval[metric_cols].mean().round(3).to_string())

# Out-of-scope manual check
print("\n" + "="*60)
print("OUT-OF-SCOPE QUESTION HANDLING")
print("="*60)
decline_phrases = [
    "only able to answer", "don't have that information",
    "not able to respond", "out of scope", "i can't",
    "cannot answer", "unable to answer",
]
for r in [r for r in test_results if r["category"] == "out-of-scope"]:
    print(f"\nQ{r['id']}: {r['question']}")
    print(f"Answer  : {r['answer']}")
    declined = any(p in r["answer"].lower() for p in decline_phrases)
    print("✅ Correctly declined" if declined else "❌ Should have declined but didn't")

# Export
df_test = pd.DataFrame(test_results)
df_test["contexts"] = df_test["contexts"].apply(lambda x: " || ".join(x) if x else "")
df_test["sources"]  = df_test["sources"].apply(lambda x: ", ".join(x) if x else "")
df_test.to_csv("test_results.csv", index=False, encoding="utf-8")
print("✅ Saved: test_results.csv")

#from google.colab import files
#files.download("test_results.csv")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


CLIP re-initialized on cuda for test set evaluation.
Test set saved: 8 questions
Amazon test set retrieval generated for 8 questions.
Running Q1: What are the features of the DJI Smart Controller?...
Running Q2: What gaming headsets are available for Xbox?...
Running Q3: Show me educational toys for kids...
Running Q4: I need something to protect my wrists while skating...
Running Q5: What RC accessories are available?...
Running Q6: What classic games are available?...
Running Q7: What is the weather like today?...
Running Q8: Who won the latest football game?...

Completed 8 queries.

CATEGORY: FACTUAL

Q1: What are the features of the DJI Smart Controller?
Expected : The DJI Smart Controller is a remote controller compatible with DJI drones.
Got      : The DJI Smart Controller features an ultra-bright 5.5" 1080P display with a screen brightness of 1000 cd/M², designed to maximize your outdoor flying 
Grounding: 0.53

Q2: What gaming headsets are available for Xbox?
Expected : The Tu

Evaluating:   0%|          | 0/24 [00:00<?, ?it/s]


PER-QUESTION RAGAS SCORES
                                             Question    Category  faithfulness  answer_relevancy  context_precision  context_recall
What are the features of the DJI Smart Controller?...     factual         0.800             0.916              1.000             1.0
      What gaming headsets are available for Xbox?...     factual         0.000             0.904              1.000             1.0
                 Show me educational toys for kids...    semantic         0.818             0.904              1.000             1.0
I need something to protect my wrists while skatin...    semantic         0.000             0.744              0.333             1.0
                What RC accessories are available?... multi-chunk         0.750             0.947              0.000             0.0
                 What classic games are available?... multi-chunk         0.000             0.942              1.000             1.0

AVERAGE SCORES BY CATEGORY
             f

In [ ]:
# ── SECTION 19: INTERACTIVE Q&A LOOP
#Reload clip
if 'clip_model' not in dir():
    device         = "cuda" if torch.cuda.is_available() else "cpu"
    clip_model     = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
    clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
    clip_model.eval()
    print(f"CLIP reloaded for interactive use on {device}")

# Fresh session for interactive use
store[SESSION_ID] = ChatMessageHistory()

print("Amazon Product Assistant — Ask me anything!")
print("The assistant remembers your previous questions.")
print("Type 'quit' to exit.\n")

while True:
    query = input("You: ").strip()
    if query.lower() in ("quit", "exit", "q"):
        print("Goodbye!")
        break
    if not query:
        continue

    result = run_rag_query(query, session_id=SESSION_ID)
    print(f"\nAssistant: {result['answer']}")
    if result["warning"]:
        print(f"\n{result['warning']}")
    if result.get("sources"):
        print("\nSources:")
        for src in result["sources"]:
            print(f"   {src}")
    print(f"\n   [Grounding score: {result.get('ground_score', 'N/A')}]")
    print("\n" + "─" * 60 + "\n")

Amazon Product Assistant — Ask me anything!
The assistant remembers your previous questions.
Type 'quit' to exit.

You: quit
Goodbye!
